# CucuGAN auf Kaggle

**Einstellungen (rechte Seitenleiste → Session options):**
- Accelerator: **GPU T4 x2**: bis zu zwei Läufe gleichzeitig, je einer pro GPU
- Internet: **an** (für `git clone`)
- Input: den Datensatz mit `collage128` hinzufügen ("Add Input")

**Ablauf:** In Zelle 1 die Läufe eintragen → alle Zellen ausführen. Für lange Läufe oben rechts
**Save Version → Save & Run All (Commit)**: läuft dann bis zu 12 h ohne offenen Browser.
Ergebnisse liegen danach im Output der Version (`/kaggle/working/CucuGAN/outputs/<lauf>/samples/`).

**Fortsetzen:** Den Output der vorigen Version als Input hinzufügen ("Add Input" → Notebook Output).
Zelle 3 findet `latest.pt` je Lauf automatisch und hängt `--resume` an.

In [ ]:
# 1) Einstellungen: ein oder zwei Läufe (je einer pro GPU): (Name, Skript, Optionen)
#    Skript: "train_cut" = CUT (ein Generator, PatchNCE)  |  "train" = CycleGAN
BRANCH = "cut"                # welcher Stand des Codes
KURZ = "--n-epochs 20 --n-epochs-decay 0"      # Kurztest

RUNS = [
    ("cut_test",     "train_cut", f"{KURZ} --amp"),                 # GPU 0: CUT wie im Paper
    ("cut_att_test", "train_cut", f"{KURZ} --amp --attention"),     # GPU 1: CUT + Attention-Maske
]

# Voller Lauf (200 Epochen, mit "Save Version"): KURZ weglassen.
# Erledigte Läufe: siehe EXPERIMENTS.md

In [ ]:
# 2) Code holen, GPUs und Daten finden
import glob, os, shutil, subprocess, time
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
if not os.path.exists("/kaggle/working/CucuGAN"):
    !git clone -q -b {BRANCH} https://github.com/Felix-uniulm/CucuGAN.git /kaggle/working/CucuGAN
%cd /kaggle/working/CucuGAN
!git log --oneline -1
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
assert len(RUNS) <= torch.cuda.device_count(), "mehr Läufe als GPUs"

DATA = next(os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/train/A", recursive=True))
print("Daten:", DATA, "|", len(os.listdir(f"{DATA}/train/A")), "Gesichter,", len(os.listdir(f"{DATA}/train/B")), "Gurken")

In [ ]:
# 3) Fortsetzen? Checkpoints früherer Outputs übernehmen (je Lauf)
resume = {}
for name, *_ in RUNS:
    prev = glob.glob(f"/kaggle/input/**/checkpoints/{name}/latest.pt", recursive=True)
    resume[name] = ""
    if prev:
        os.makedirs(f"checkpoints/{name}", exist_ok=True)
        for f in glob.glob(os.path.join(os.path.dirname(prev[0]), "*.pt")):
            shutil.copy(f, f"checkpoints/{name}/")
        prev_out = prev[0].replace(f"checkpoints/{name}/latest.pt", f"outputs/{name}")
        if os.path.exists(prev_out):
            shutil.copytree(prev_out, f"outputs/{name}", dirs_exist_ok=True)
        resume[name] = "--resume"
    print(name, "->", "setze fort" if resume[name] else "neu")

In [ ]:
# 4) Training: alle Läufe gleichzeitig starten, jeder auf seiner GPU, dann warten
os.makedirs("logs", exist_ok=True)
procs = []
for gpu, (name, script, extra) in enumerate(RUNS):
    cmd = f"python src/{script}.py --name {name} --data {DATA} --num-workers 2 {extra} {resume[name]}"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))   # jeder Lauf sieht nur "seine" GPU
    log = open(f"logs/{name}.log", "w")
    procs.append((name, subprocess.Popen(cmd.split(), env=env, stdout=log, stderr=subprocess.STDOUT), log))
    print(f"GPU {gpu}: {cmd}")

def last_epoch_line(name):
    lines = [l for l in open(f"logs/{name}.log", errors="ignore").read().replace("\r", "\n").splitlines()
             if "fertig" in l or "Error" in l or "Traceback" in l]
    return lines[-1].strip() if lines else "(startet ...)"

while any(p.poll() is None for _, p, _ in procs):
    time.sleep(300)                                          # alle 5 min Zwischenstand
    for name, p, _ in procs:
        print(f"[{time.strftime('%H:%M')}] {name}: {last_epoch_line(name)}", flush=True)
for name, p, log in procs:
    log.close()
    print(f"{name}: beendet mit Code {p.returncode} | {last_epoch_line(name)}")
    if p.returncode != 0:
        print(open(f"logs/{name}.log", errors="ignore").read()[-3000:])

In [ ]:
# 5) Letzte Kontrollbilder (und bei --attention die Maske) je Lauf anzeigen
from IPython.display import Image, display
for name, *_ in RUNS:
    samples = sorted(glob.glob(f"outputs/{name}/samples/epoch_*.png"))
    print(f"=== {name}: {len(samples)} Kontrollbilder; zuletzt:", samples[-1] if samples else "-")
    if samples:
        display(Image(samples[-1]))
    masks = sorted(glob.glob(f"outputs/{name}/samples/mask_*.png"))
    if masks:
        print("Maske (hell = wird zu Gurke):", masks[-1])
        display(Image(masks[-1]))